In [1]:
import numpy as np
import cupy as cp

In [2]:
x_gpu = cp.array([1,2,3])

In [3]:
x_cpu = np.array([1, 2, 3])
l2_cpu = np.linalg.norm(x_cpu)
l2_cpu

np.float64(3.7416573867739413)

In [4]:
x_gpu = cp.array([1, 2, 3])
l2_gpu = cp.linalg.norm(x_gpu)
l2_gpu

array(3.74165739)

In [5]:
x_on_gpu0 = cp.array([1, 2, 3, 4, 5])

## Current Device

In [9]:
cp.cuda.Device(0)

<CUDA Device 0>

In [8]:
with cp.cuda.Device(0):
   x_on_gpu1 = cp.array([1, 2, 3, 4, 5])
x_on_gpu0 = cp.array([1, 2, 3, 4, 5])

In [10]:
with cp.cuda.Device(0):
    x = cp.array([1,2,3,4,65])
x.device

<CUDA Device 0>

## Data Transfer

In [11]:
x_cpu = np.array([1,2,3,4])
x_gpu = cp.asarray(x_cpu)

In [13]:
with cp.cuda.Device(0):
    x_gpu_0 = cp.ndarray([1, 2, 3])  # create an array in GPU 0


## Move array from a device to the host

In [14]:
x_gpu = cp.array([1, 2, 3])  # create an array in the current device
x_cpu = cp.asnumpy(x_gpu)  # move the array to the host.

     We can also use cupy.ndarray.get():

In [15]:
x_cpu = x_gpu.get()

## Memory management

#### How to write CPU/GPU agnostic code

In [16]:
# Stable implementation of log(1 + exp(x))
def softplus(x):
    xp = cp.get_array_module(x)  # 'xp' is a standard usage in the community
    print("Using:", xp.__name__)
    return xp.maximum(0, x) + xp.log1p(xp.exp(-abs(x)))

In [29]:
sample = softplus(x_cpu)
sample

Using: numpy


array([1.31326169, 2.12692801, 3.04858735])

In [30]:
sample = softplus(x_gpu)
sample

Using: cupy


array([1.31326169, 2.12692801, 3.04858735])

When you need to manipulate CPU and GPU arrays, an explicit data transfer may be required to move them to the same location – either CPU or GPU. For this purpose, CuPy implements two sister methods called cupy.asnumpy() and cupy.asarray(). Here is an example that demonstrates the use of both methods:

In [17]:
x_cpu = np.array([1, 2, 3])
y_cpu = np.array([4, 5, 6])
x_cpu + y_cpu

array([5, 7, 9])

This operation fails due to device conflict

In [18]:
x_gpu = cp.asarray(x_cpu)
x_gpu + y_cpu

TypeError: Unsupported type <class 'numpy.ndarray'>

In [28]:
x_gpu.device

<CUDA Device 0>

but using the 
 cp.asnumpy() moves the tensor to cpu

In [19]:
cp.asnumpy(x_gpu) + y_cpu

array([5, 7, 9])

In [20]:
cp.asnumpy(x_gpu).device

'cpu'

In [24]:
cp.asnumpy(x_gpu) + cp.asnumpy(y_cpu)

array([5, 7, 9])

but using the 
 cp.asarray() moves the tensor to Gpu

In [25]:
x_gpu + cp.asarray(y_cpu)

array([5, 7, 9])

In [26]:
cp.asarray(y_cpu).device

<CUDA Device 0>

In [27]:
cp.asarray(x_gpu) + cp.asarray(y_cpu)

array([5, 7, 9])

# [Custom kernels](https://docs.cupy.dev/en/stable/reference/kernel.html)

## User-Defined Kernels

CuPy provides easy ways to define three types of CUDA kernels: elementwise kernels, reduction kernels, and raw kernels. In this documentation, we describe how to define and call each kernel.

### Basics of elementwise kernels

An elementwise kernel can be defined by the ElementwiseKernel class. The instance of this class defines a CUDA kernel which can be invoked by the __call__ method of this instance.

A definition of an elementwise kernel consists of four parts: an input argument list, an output argument list, a loop body code, and the kernel name. For example, a kernel that computes a squared difference 
 is defined as follows:

In [31]:
squared_diff = cp.ElementwiseKernel(
   'float32 x, float32 y',
   'float32 z',
   'z = (x - y) * (x - y)',
   'squared_diff')

The argument lists consist of comma-separated argument definitions. Each argument definition consists of a type specifier and an argument name. Names of NumPy data types can be used as type specifiers.

###  Note
      n, i, and names starting with an underscore _ are reserved for the internal use.

In [33]:
x = cp.arange(10, dtype=np.float32).reshape(2, 5)
y = cp.arange(5, dtype=np.float32)

print(f'X: {x}')
print(f'Y: {y}')

squared_diff(x, y)

X: [[0. 1. 2. 3. 4.]
 [5. 6. 7. 8. 9.]]
Y: [0. 1. 2. 3. 4.]


array([[ 0.,  0.,  0.,  0.,  0.],
       [25., 25., 25., 25., 25.]], dtype=float32)

In [34]:
squared_diff(x, 5)

array([[25., 16.,  9.,  4.,  1.],
       [ 0.,  1.,  4.,  9., 16.]], dtype=float32)

## Reduction kernels

Reduction kernels can be defined by the ReductionKernel class. We can use it by defining four parts of the kernel code:

Identity value: This value is used for the initial value of reduction.

Mapping expression: It is used for the pre-processing of each element to be reduced.

Reduction expression: It is an operator to reduce the multiple mapped values. The special variables a and b are used for its operands.

Post mapping expression: It is used to transform the resulting reduced values. The special variable a is used as its input. Output should be written to the output parameter.

ReductionKernel class automatically inserts other code fragments that are required for an efficient and flexible reduction implementation.

For example, L2 norm along specified axes can be written as follows:

In [35]:
l2norm_kernel = cp.ReductionKernel(
    'T x',  # input params
    'T y',  # output params
    'x * x',  # map
    'a + b',  # reduce
    'y = sqrt(a)',  # post-reduction map
    '0',  # identity value
    'l2norm'  # kernel name
)
x = cp.arange(10, dtype=np.float32).reshape(2, 5)
l2norm_kernel(x, axis=1)

array([ 5.477226 , 15.9687195], dtype=float32)

## Raw kernels

Raw kernels can be defined by the RawKernel class. By using raw kernels, you can define kernels from raw CUDA source.

RawKernel object allows you to call the kernel with CUDA’s cuLaunchKernel interface. In other words, you have control over grid size, block size, shared memory size, and stream.

### Note

Unlike ElementwiseKernel, RawKernel ignores any views on CuPy arrays. You are responsible for handling strides manually. For example, passing matrix.T will be treated as if you passed matrix.

In [38]:
add_kernel = cp.RawKernel(r'''
extern "C" __global__
void my_add(const float* x1, const float* x2, float* y) {
    int tid = blockDim.x * blockIdx.x + threadIdx.x;
    y[tid] = x1[tid] + x2[tid];
}
''', 'my_add')
x1 = cp.arange(25, dtype=cp.float32).reshape(5, 5)
x2 = cp.arange(25, dtype=cp.float32).reshape(5, 5)
y = cp.zeros((5, 5), dtype=cp.float32)
add_kernel((5,), (5,), (x1, x2, y))  # grid, block and arguments
y


array([[ 0.,  2.,  4.,  6.,  8.],
       [10., 12., 14., 16., 18.],
       [20., 22., 24., 26., 28.],
       [30., 32., 34., 36., 38.],
       [40., 42., 44., 46., 48.]], dtype=float32)

In [39]:
complex_kernel = cp.RawKernel(r'''
#include <cupy/complex.cuh>
extern "C" __global__
void my_func(const complex<float>* x1, const complex<float>* x2,
             complex<float>* y, float a) {
    int tid = blockDim.x * blockIdx.x + threadIdx.x;
    y[tid] = x1[tid] + a * x2[tid];
}
''', 'my_func')
x1 = cp.arange(25, dtype=cp.complex64).reshape(5, 5)
x2 = 1j*cp.arange(25, dtype=cp.complex64).reshape(5, 5)
y = cp.zeros((5, 5), dtype=cp.complex64)
complex_kernel((5,), (5,), (x1, x2, y, cp.float32(2.0)))  # grid, block and arguments
y

array([[ 0. +0.j,  1. +2.j,  2. +4.j,  3. +6.j,  4. +8.j],
       [ 5.+10.j,  6.+12.j,  7.+14.j,  8.+16.j,  9.+18.j],
       [10.+20.j, 11.+22.j, 12.+24.j, 13.+26.j, 14.+28.j],
       [15.+30.j, 16.+32.j, 17.+34.j, 18.+36.j, 19.+38.j],
       [20.+40.j, 21.+42.j, 22.+44.j, 23.+46.j, 24.+48.j]],
      dtype=complex64)

## Custom user types

It is possible to use custom types (composite types such as structures and structures of structures) as kernel arguments by defining a custom NumPy dtype. When doing this, it is your responsibility to match the host and device structure memory layout. The CUDA standard guarantees that the size of fundamental types on the host and device always match. It may, however, impose device alignment requirements on composite types. This means that for composite types the struct member offsets may be different from what you might expect.

When a kernel argument is passed by value, the CUDA driver will copy exactly sizeof(param_type) bytes starting from the beginning of the NumPy object data pointer, where param_type is the parameter type in your kernel. You have to match param_type’s memory layout (ex: size, alignment and struct padding/packing) by defining a corresponding NumPy dtype.

For builtin CUDA vector types such as int2 and double4 and other packed structures with named members you can directly define such NumPy dtypes as the following:

In [40]:
import numpy as np
names = ['x', 'y', 'z']
types = [np.float32]*3
float3 = np.dtype({'names': names, 'formats': types})
arg = np.random.rand(3).astype(np.float32).view(float3)
print(arg)  


[(0.10059293, 0.01247013, 0.24802902)]


In [41]:

arg['x'] = 42.0
print(arg)  


[(42., 0.01247013, 0.24802902)]


## Raw modules

For dealing a large raw CUDA source or loading an existing CUDA binary, the RawModule class can be more handy. It can be initialized either by a CUDA source code, or by a path to the CUDA binary. It accepts most of the arguments as in RawKernel. The needed kernels can then be retrieved by calling the get_function() method, which returns a RawKernel instance that can be invoked as discussed above.

In [42]:
loaded_from_source = r'''
extern "C"{

__global__ void test_sum(const float* x1, const float* x2, float* y, \
                         unsigned int N)
{
    unsigned int tid = blockDim.x * blockIdx.x + threadIdx.x;
    if (tid < N)
    {
        y[tid] = x1[tid] + x2[tid];
    }
}

__global__ void test_multiply(const float* x1, const float* x2, float* y, \
                              unsigned int N)
{
    unsigned int tid = blockDim.x * blockIdx.x + threadIdx.x;
    if (tid < N)
    {
        y[tid] = x1[tid] * x2[tid];
    }
}

}'''
module = cp.RawModule(code=loaded_from_source)
ker_sum = module.get_function('test_sum')
ker_times = module.get_function('test_multiply')
N = 10
x1 = cp.arange(N**2, dtype=cp.float32).reshape(N, N)
x2 = cp.ones((N, N), dtype=cp.float32)
y = cp.zeros((N, N), dtype=cp.float32)
ker_sum((N,), (N,), (x1, x2, y, N**2))   # y = x1 + x2
assert cp.allclose(y, x1 + x2)
ker_times((N,), (N,), (x1, x2, y, N**2)) # y = x1 * x2
assert cp.allclose(y, x1 * x2)

## Kernel fusion

cupy.fuse() is a decorator that fuses functions. This decorator can be used to define an elementwise or reduction kernel more easily than ElementwiseKernel or ReductionKernel.

By using this decorator, we can define the squared_diff kernel as follows:

In [43]:
@cp.fuse()
def squared_diff(x, y):
    return (x - y) * (x - y)

The above kernel can be called on either scalars, NumPy arrays or CuPy arrays likes the original function.

In [44]:
x_cp = cp.arange(10)
y_cp = cp.arange(10)[::-1]
squared_diff(x_cp, y_cp)

array([81, 49, 25,  9,  1,  1,  9, 25, 49, 81])

In [45]:
x_np = np.arange(10)
y_np = np.arange(10)[::-1]
squared_diff(x_np, y_np)

array([81, 49, 25,  9,  1,  1,  9, 25, 49, 81])

At the first function call, the fused function analyzes the original function based on the abstracted information of arguments (e.g. their dtypes and ndims) and creates and caches an actual CUDA kernel. From the second function call with the same input types, the fused function calls the previously cached kernel, so it is highly recommended to reuse the same decorated functions instead of decorating local functions that are defined multiple times.

     cupy.fuse() also supports a simple reduction kernel.

In [46]:
@cp.fuse()
def sum_of_products(x, y):
    return cp.sum(x * y, axis = -1)

In [47]:
@cp.fuse(kernel_name='squared_diff')
def squared_diff(x, y):
    return (x - y) * (x - y)

## Note

Currently, cupy.fuse() can fuse only simple elementwise and reduction operations. Most other routines (e.g. cupy.matmul(), cupy.reshape()) are not supported.

## JIT kernel definition

The cupyx.jit.rawkernel decorator can create raw CUDA kernels from Python functions.

In this section, a Python function wrapped with the decorator is called a target function.

A target function consists of elementary scalar operations, and users have to manage how to parallelize them. CuPy’s array operations which automatically parallelize operations (e.g., add(), sum()) are not supported. If a custom kernel based on such array functions is desired, please refer to the Kernel fusion section.

### Basic Usage
Here is a short example for how to write a cupyx.jit.rawkernel to copy the values from x to y using a grid-stride loop:

In [49]:
from cupyx import jit

@jit.rawkernel()
def elementwise_copy(x, y, size):
    tid = jit.blockIdx.x * jit.blockDim.x + jit.threadIdx.x
    ntid = jit.gridDim.x * jit.blockDim.x
    for i in range(tid, size, ntid):
        y[i] = x[i]

size = cp.uint32(2 ** 22)
x = cp.random.normal(size=(size,), dtype=cp.float32)
y = cp.empty((size,), dtype=cp.float32)

elementwise_copy((128,), (1024,), (x, y, size))  # RawKernel style
assert (x == y).all()

elementwise_copy[128, 1024](x, y, size)  #  Numba style
assert (x == y).all()

x:\VS_CODE\LLM_track\.venv\Lib\site-packages\cupyx\jit\_interface.py:247: FutureWarning: cupyx.jit.rawkernel is experimental. The interface can change in the future.
  cupy._util.experimental('cupyx.jit.rawkernel')


Both styles to launch the kernel, as shown above, are supported. The first two entries are the grid and block sizes, respectively. grid ( RawKernel style (128,) or Numba style [128]) is the sizes of the grid, i.e., the numbers of blocks in each dimension; block ((1024,) or [1024]) is the dimensions of each thread block, please refer to [cupyx.jit._interface._JitRawKernel](https://docs.cupy.dev/en/stable/reference/generated/cupyx.jit._interface._JitRawKernel.html#cupyx.jit._interface._JitRawKernel) for details. Launching a CUDA kernel on a GPU with pre-determined grid/block sizes requires basic understanding in the [CUDA Programming Model](https://developer.nvidia.com/blog/cuda-refresher-cuda-programming-model/).

The compilation will be deferred until the first function call. CuPy’s JIT compiler infers the types of arguments at call time and will cache the compiled kernels for speeding up any subsequent calls.

See [Custom kernels](https://docs.cupy.dev/en/stable/reference/kernel.html) for a full list of API.

### Basic Design
CuPy’s JIT compiler generates CUDA code via Python AST. We decided not to use Python bytecode to analyze the target function to avoid performance degradation. The CUDA source code generated from the Python bytecode will not effectively be optimized by the CUDA compiler, because for-loops and other control statements of the target function are fully transformed to jump instruction when converting the target function to bytecode.

### Typing rule
The types of local variables are inferred at the first assignment in the function. The first assignment must be done at the top-level of the function; in other words, it must not be in if/else bodies or for-loops.

### Limitations
JIT does not work inside Python’s interactive interpreter (REPL) as the compiler needs to get the source code of the target function.